# Guia de estudio para la evaluacion

Limpieza del **Healthcare Dataset** (55.500 admisiones). Repaso paso a paso para saber defender el codigo y las decisiones tomadas, construido sobre el codigo real de la carpeta `Codigo Limpieza de Datos/`.

## 1. Que hace el proyecto

Limpia el Healthcare Dataset de Kaggle: quita duplicados, normaliza texto y marca valores imposibles (facturas negativas) sin tocar el monto. El resultado se guarda en `healthcare_dataset_limpio.csv`.

## 2. Arquitectura por capas (multiservicios)

Cada capa tiene una responsabilidad y un solo sentido de dependencia (main -> servicios -> repositorios/utilidades).

| Capa | Archivo | Responsabilidad |
|---|---|---|
| Orquestador | `main.py` | `PipelineLimpieza` define el **orden** de las operaciones |
| Repositorio | `repositorios/cargador.py` | Acceso a datos: lee el CSV (`pd.read_csv`) |
| Servicios (transformacion) | `servicios/limpieza.py` | Duplicados, texto, marcado de facturas negativas |
| Servicios (calidad) | `servicios/calidad.py` | Diagnostico, outliers, validacion |
| Utilidades | `utilidades/constantes.py` | Rutas y nombres de columnas |

## 3. Orden del pipeline y por que

1. Cargar el CSV
2. Diagnostico del dataset crudo (linea base)
3. Quitar duplicados
4. Normalizar texto
5. Marcar facturas negativas (no se imputa nada)
6. Revisar outliers (IQR)
7. Diagnostico del dataset limpio
8. Validar
9. Exportar CSV

**Claves para defender el orden:**

- Los **duplicados se quitan primero** porque si no, las medianas y conteos posteriores cuentan filas doble y sesgan todo.
- Las facturas negativas se **marcan** y quedan **excluidas** del análisis de outliers (IQR), porque un negativo no es un outlier estadístico sino un error de negocio (ver sección 7).

## 4. Repaso de cada archivo

### `utilidades/constantes.py` — el "mapa"

```python
from pathlib import Path

CARPETA_UTILIDADES = Path(__file__).resolve().parent   # utilidades/
CARPETA_CODIGO = CARPETA_UTILIDADES.parent             # Codigo Limpieza de Datos/
CARPETA_PROYECTO = CARPETA_CODIGO.parent               # raiz (mineria/)

ARCHIVO_CRUDO = CARPETA_PROYECTO / "healthcare_dataset_original.csv"
ARCHIVO_LIMPIO = CARPETA_PROYECTO / "healthcare_dataset_limpio.csv"

COLUMNAS_NUMERICAS = ["Age", "Billing Amount"]
COLUMNAS_TEXTO = ["Name", "Doctor", "Hospital"]
```

- `Path(__file__)` = archivo actual. `.resolve()` = ruta absoluta (no depende de donde se ejecute). `.parent` = carpeta que lo contiene.
- Se suben dos niveles porque constantes.py está 2 carpetas adentro de la raíz.
- Así el código encuentra los CSV sin importar el directorio de trabajo (a diferencia del notebook, que usa `Path.cwd()`).

### `repositorios/cargador.py` — leer los datos

```python
import pandas as pd

class CargadorDataset:
    def __init__(self, archivo):
        self.archivo = archivo

    def cargar(self):
        return pd.read_csv(self.archivo)
```

- `CargadorDataset(ARCHIVO_CRUDO)` guarda la ruta en `self.archivo`.
- `.cargar()` lee el CSV y devuelve un **DataFrame** de pandas.
- Resultado real: 55.500 filas x 15 columnas.

### `main.py` — el supervisor

`__init__` solo **prepara** el equipo (instancia todos los servicios). `ejecutar()` llama a cada servicio en orden.

**Patrón clave:** los transformadores (duplicados, texto, imposibles) **reciben el df y devuelven uno nuevo** (`df = servicio.metodo(df)`); los analizadores (diagnóstico, outliers, validación) **solo imprimen**.

```python
if __name__ == "__main__":
    PipelineLimpieza(ARCHIVO_CRUDO, ARCHIVO_LIMPIO).ejecutar()
```

- `__name__ == "__main__"`: solo se ejecuta si el archivo se corre directamente (`python "Codigo Limpieza de Datos/main.py"`), no si se importa.

### `servicios/limpieza.py` — transformaciones

**TratadorDuplicados:**

```python
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
```

- `drop_duplicates()` elimina filas iguales en todas las columnas.
- `reset_index(drop=True)` renumera el índice 0,1,2... (al borrar filas quedan huecos).
- Quita **534** filas (55500 -> 54966).

**NormalizadorTexto:**

```python
df[columna] = df[columna].str.strip().str.title()
```

- `.str` habilita operaciones de texto. `.strip()` quita espacios iniciales/finales. `.title()` mayúscula inicial en cada palabra: `"bObBy jAcKsOn"` -> `"Bobby Jackson"`.
- Motivo: un mismo nombre escrito distinto no coincide en búsquedas ni agrupaciones.

### `servicios/calidad.py` — análisis y validación

**Diagnosticador:** imprime filas, duplicados, nulos y `describe()` (count, mean, std, min, cuartiles, max).

**AnalizadorOutliers (IQR / regla de Tukey):**

```python
q1, q3 = df[columna].quantile([0.25, 0.75])
iqr = q3 - q1
limiteInferior = q1 - 1.5 * iqr
limiteSuperior = q3 + 1.5 * iqr
fuera = ((df[columna] < limiteInferior) | (df[columna] > limiteSuperior)).sum()
```

- Q1 = valor bajo el cual está el 25% de los datos; Q3 = 75%. IQR = Q3 - Q1.
- Outlier = todo lo que quede fuera de `Q1 - 1.5*IQR` y `Q3 + 1.5*IQR`.
- En tus datos: 0 fuera de rango en Age y Billing Amount. **Este paso solo imprime, no modifica el df.**

**MarcadorFacturasNegativas (lo más importante):**

```python
negativos = df["Billing Amount"] < 0   # máscara booleana (True donde es negativo)
df[columnaFlag] = negativos            # nueva columna que marca las 106
```

- La máscara booleana es una Serie de True/False; `.sum()` cuenta los True (106).
- `df[columnaFlag] = negativos` agrega la columna `Factura_Negativa` con True en las negativas y False en el resto.
- **No se toca el monto**: es dinero, un dato sensible; inventar un valor (mediana, media o cero) fabricaría una factura que la empresa nunca registró. El dato se conserva marcado para auditoría y queda fuera de los cálculos.

**Validador:** checks finales: duplicados, nulos, facturas negativas (todas marcadas, con el monto original intacto) y edades fuera de rango (0-120).

## 5. Números para memorizar

| Métrica | Valor |
|---|---|
| Filas originales | 55.500 |
| Filas limpias | 54.966 |
| Duplicados eliminados | 534 |
| Nulos (siempre) | 0 |
| Facturas negativas en el limpio | 106 (marcadas) |
| Imputaciones de dinero | 0 (el monto original se conserva) |
| Límites IQR Age | [-14,50 , 117,50] -> 0 fuera |
| Límites IQR Billing Amount (válidos) | [-23.521,23 , 74.668,04] -> 0 fuera |
| Edad mínima / máxima | 13 / 89 |
| Al final | 0 duplicados, 0 nulos, 0 edades fuera de rango, 106 negativas marcadas (monto intacto) |

## 6. Decisiones y cómo defenderlas

1. **Orden: duplicados primero.** Sin eso, medianas y conteos cuentan doble.

2. **No imputar dinero.** En datos financieros no se inventan valores: reemplazar un monto con mediana, media o cero fabrica una factura que la empresa nunca registró y puede generar pérdidas significativas. Por eso el monto se conserva intacto y solo se marca.

3. **No borrar las filas con factura negativa.** El resto del registro (paciente, diagnóstico, admisión) sigue siendo útil; el monto es incorrecto, se marca y se excluye de los cálculos, quedando disponible para auditoría.

4. **El error es MCAR, pero eso no autoriza a imputar.** Se verificó que los negativos están repartidos igual que el dataset completo por tipo de admisión, aseguradora y condición médica: el error es aleatorio e independiente (MCAR). Eso solo confirma que no hay forma de derivar el valor correcto; en dinero la respuesta es marcar y excluir, no inventar.

5. **Rutas por `__file__` y no por `cwd()`.** El notebook depende de dónde lo abras; el código en constantes.py no, usa la ruta real del archivo.

6. **IQR vs regla de negocio.** Sobre los valores válidos, el límite inferior de Billing Amount dio -23521.23 y el mínimo real es 9.24: el IQR nunca iba a detectar los negativos porque no son outliers estadísticos. Ese problema solo lo atrapa la regla de negocio (una factura no puede ser negativa). Lección: la estadística sola no basta.

## 7. Conceptos teóricos

- **MCAR vs MAR vs MNAR:** en tu caso MCAR (el error no depende de ninguna variable observada). Pero que sea aleatorio **no** autoriza a imputar dinero: se marca y se excluye.
- **Datos faltantes:** no solo NaN; los valores "imposibles" (negativos) son un problema de calidad y, al ser dinero, se resuelven marcando y excluyendo, no inventando valores.
- **IQR:** robusto, no asume normalidad (a diferencia de la desviación estándar).
- **`__init__.py`:** convierte carpetas en paquetes Python importables.

## 8. Respuesta de una frase para la evaluación

"Diseñé una limpieza por capas: primero diagnostico el crudo, después, en orden, quito duplicados (para no sesgar las cuentas), normalizo texto (para unificar nombres), marco las 106 facturas negativas en una columna aparte sin tocar el monto, porque es dinero y no se inventan valores (el error resultó aleatorio, MCAR), reviso outliers con IQR sobre los valores válidos (no había) y valido que queden 0 duplicados, 0 nulos y las 106 facturas marcadas con su monto original intacto en las 54.966 filas finales."

## 9. Resumen del código en vivo

Este notebook corre el pipeline real de un solo golpe: imprime el diagnóstico comparativo, los límites IQR y las validaciones finales. (Es exactamente lo que hace `Codigo Limpieza de Datos/main.py`.)

In [ ]:
import sys
from pathlib import Path

CARPETA_CODIGO = Path.cwd() / "Codigo Limpieza de Datos"
if str(CARPETA_CODIGO) not in sys.path:
    sys.path.append(str(CARPETA_CODIGO))

from repositorios.cargador import CargadorDataset
from utilidades.constantes import ARCHIVO_CRUDO, COLUMNAS_NUMERICAS, COLUMNAS_TEXTO, COLUMNA_FLAG_FACTURA_NEGATIVA
from servicios.calidad import Diagnosticador, AnalizadorOutliers, Validador
from servicios.limpieza import MarcadorFacturasNegativas, NormalizadorTexto, TratadorDuplicados

df = CargadorDataset(ARCHIVO_CRUDO).cargar()
diagnosticador = Diagnosticador(COLUMNAS_NUMERICAS, COLUMNA_FLAG_FACTURA_NEGATIVA)
diagnosticador.diagnosticar(df, "Dataset crudo")

df = TratadorDuplicados().quitar(df)
df = NormalizadorTexto(COLUMNAS_TEXTO).normalizar(df)
df = MarcadorFacturasNegativas(COLUMNA_FLAG_FACTURA_NEGATIVA).marcar(df)

AnalizadorOutliers(COLUMNAS_NUMERICAS, COLUMNA_FLAG_FACTURA_NEGATIVA).revisarIQR(df)

print("-----")
diagnosticador.diagnosticar(df, "Dataset limpio")
Validador(COLUMNA_FLAG_FACTURA_NEGATIVA).validar(df)
print("Filas finales:", len(df))

=== Dataset crudo ===
Filas: 55500
Duplicados: 534
Nulos: 0
            Age  Billing Amount
count  55500.00        55500.00
mean      51.54        25539.32
std       19.60        14211.45
min       13.00        -2008.49
25%       35.00        13241.22
50%       52.00        25538.07
75%       68.00        37820.51
max       89.00        52764.28
Duplicados eliminados: 534
Facturas negativas marcadas: 106
El monto original se conserva, no se imputa ningún valor.
Age: limites [-14.50, 117.50], fuera de rango: 0
Billing Amount: limites [-23521.23, 74668.04], fuera de rango: 0
-----
=== Dataset limpio ===
Filas: 54966
Duplicados: 0
Nulos: 0
Facturas negativas marcadas: 106
            Age  Billing Amount
count  54860.00        54860.00
mean      51.53        25594.63
std       19.61        14175.87
min       13.00            9.24
25%       35.00        13299.75
50%       52.00        25593.87
75%       68.00        37847.07
max       89.00        52764.28
Duplicados: 0
Nulos: 0
Facturas ne